In [11]:
import os
import joblib
import numpy as np

from tqdm import tqdm

from sklearn.svm import SVC
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from sklearn.model_selection import (
    StratifiedKFold,
    ParameterGrid
)

In [4]:
FEATURE_DIR = r"C:\Users\AmrAhmed\Desktop\Level 3\Second Term\Pattern Recognition\Project\Drowsiness-Project\feature_extraction\extracted_features"

X_train_path = os.path.join(FEATURE_DIR, "X_train.npy")
X_test_path  = os.path.join(FEATURE_DIR, "X_test.npy")
y_train_path = os.path.join(FEATURE_DIR, "y_train.npy")
y_test_path  = os.path.join(FEATURE_DIR, "y_test.npy")
scaler_path  = os.path.join(FEATURE_DIR, "scaler.pkl")
model_path   = os.path.join(FEATURE_DIR, "svm_model.pkl")

In [5]:
X_train = np.load(X_train_path)
X_test  = np.load(X_test_path)
y_train = np.load(y_train_path)
y_test  = np.load(y_test_path)

y_train = y_train.ravel()
y_test = y_test.ravel()

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape :", y_test.shape)

X_train shape: (22420, 1280)
X_test shape : (5605, 1280)
y_train shape: (22420,)
y_test shape : (5605,)


In [6]:
scaler = joblib.load(scaler_path)

X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

print("Scaling done successfully.")

Scaling done successfully.


In [7]:
svm = SVC(class_weight="balanced")

param_grid = {
    "C": [0.1, 1, 10],
    "kernel": ["linear", "rbf"],
    "gamma": ["scale", "auto"]
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

In [9]:
grid_search = GridSearchCV(
    estimator=svm,
    param_grid=param_grid,
    scoring="accuracy",
    cv=cv,
    n_jobs=-1,
    verbose=2
)

print("Starting Grid Search...")
grid_search.fit(X_train, y_train)

print("\nBest Parameters:", grid_search.best_params_)
print("Best CV Accuracy:", grid_search.best_score_)

Starting Grid Search...
Fitting 3 folds for each of 12 candidates, totalling 36 fits


KeyboardInterrupt: 

In [12]:
param_grid = {
    "C": [0.1, 1, 10],
    "kernel": ["linear", "rbf"],
    "gamma": ["scale", "auto"]
}

grid_list = list(ParameterGrid(param_grid))

print("Total Configurations:", len(grid_list))

Total Configurations: 12


In [13]:
kf = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

In [14]:
best_score = 0
best_params = None
best_model = None

results = []

for params in tqdm(grid_list, desc="Grid Search"):

    fold_scores = []

    base_model = SVC(
        **params,
        class_weight="balanced"
    )

    for train_idx, val_idx in tqdm(
        kf.split(X_train, y_train),
        total=kf.get_n_splits(),
        leave=False,
        desc=f"Folds {params}"
    ):
        X_tr = X_train[train_idx]
        X_val = X_train[val_idx]
        y_tr = y_train[train_idx]
        y_val = y_train[val_idx]

        model = clone(base_model)
        model.fit(X_tr, y_tr)

        y_pred = model.predict(X_val)
        acc = accuracy_score(y_val, y_pred)

        fold_scores.append(acc)
    mean_score = np.mean(fold_scores)

    results.append({
        "params": params,
        "score": mean_score
    })
    print(f"\nParams: {params}")
    print(f"Mean Accuracy: {mean_score:.4f}")

    if mean_score > best_score:
        best_score = mean_score
        best_params = params
        best_model = clone(base_model)

Grid Search:   0%|          | 0/12 [00:00<?, ?it/s]
Folds {'C': 0.1, 'gamma': 'scale', 'kernel': 'linear'}:   0%|          | 0/3 [00:00<?, ?it/s]
Folds {'C': 0.1, 'gamma': 'scale', 'kernel': 'linear'}:  33%|███▎      | 1/3 [00:11<00:23, 11.70s/it]
Folds {'C': 0.1, 'gamma': 'scale', 'kernel': 'linear'}:  67%|██████▋   | 2/3 [00:22<00:11, 11.06s/it]
Folds {'C': 0.1, 'gamma': 'scale', 'kernel': 'linear'}: 100%|██████████| 3/3 [00:33<00:00, 10.94s/it]
Grid Search:   8%|▊         | 1/12 [00:33<06:04, 33.11s/it]                                          


Params: {'C': 0.1, 'gamma': 'scale', 'kernel': 'linear'}
Mean Accuracy: 0.9946



Folds {'C': 0.1, 'gamma': 'scale', 'kernel': 'rbf'}:   0%|          | 0/3 [00:00<?, ?it/s]
Folds {'C': 0.1, 'gamma': 'scale', 'kernel': 'rbf'}:  33%|███▎      | 1/3 [01:52<03:44, 112.40s/it]
Folds {'C': 0.1, 'gamma': 'scale', 'kernel': 'rbf'}:  67%|██████▋   | 2/3 [03:46<01:53, 113.65s/it]
Folds {'C': 0.1, 'gamma': 'scale', 'kernel': 'rbf'}: 100%|██████████| 3/3 [05:38<00:00, 112.58s/it]
Grid Search:  17%|█▋        | 2/12 [06:11<35:25, 212.59s/it]                                       


Params: {'C': 0.1, 'gamma': 'scale', 'kernel': 'rbf'}
Mean Accuracy: 0.9744



Folds {'C': 0.1, 'gamma': 'auto', 'kernel': 'linear'}:   0%|          | 0/3 [00:00<?, ?it/s]
Folds {'C': 0.1, 'gamma': 'auto', 'kernel': 'linear'}:  33%|███▎      | 1/3 [00:10<00:20, 10.30s/it]
Folds {'C': 0.1, 'gamma': 'auto', 'kernel': 'linear'}:  67%|██████▋   | 2/3 [00:20<00:10, 10.44s/it]
Folds {'C': 0.1, 'gamma': 'auto', 'kernel': 'linear'}: 100%|██████████| 3/3 [00:30<00:00, 10.27s/it]
Grid Search:  25%|██▌       | 3/12 [06:42<19:26, 129.63s/it]                                        

...


Params: {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}
Mean Accuracy: 0.9943



Folds {'C': 10, 'gamma': 'auto', 'kernel': 'linear'}:   0%|          | 0/3 [00:00<?, ?it/s]
Folds {'C': 10, 'gamma': 'auto', 'kernel': 'linear'}:  33%|███▎      | 1/3 [00:08<00:17,  8.97s/it]
Folds {'C': 10, 'gamma': 'auto', 'kernel': 'linear'}:  67%|██████▋   | 2/3 [00:17<00:08,  8.98s/it]
Folds {'C': 10, 'gamma': 'auto', 'kernel': 'linear'}: 100%|██████████| 3/3 [00:26<00:00,  8.86s/it]
Grid Search:  92%|█████████▏| 11/12 [49:51<03:29, 209.25s/it]                                      


Params: {'C': 10, 'gamma': 'auto', 'kernel': 'linear'}
Mean Accuracy: 0.9946



Folds {'C': 10, 'gamma': 'auto', 'kernel': 'rbf'}:   0%|          | 0/3 [00:00<?, ?it/s]
Folds {'C': 10, 'gamma': 'auto', 'kernel': 'rbf'}:  33%|███▎      | 1/3 [07:51<15:43, 471.57s/it]
Folds {'C': 10, 'gamma': 'auto', 'kernel': 'rbf'}:  67%|██████▋   | 2/3 [16:03<08:03, 483.39s/it]
Folds {'C': 10, 'gamma': 'auto', 'kernel': 'rbf'}: 100%|██████████| 3/3 [23:54<00:00, 477.88s/it]
Grid Search: 100%|██████████| 12/12 [1:13:46<00:00, 368.84s/it]                                  


Params: {'C': 10, 'gamma': 'auto', 'kernel': 'rbf'}
Mean Accuracy: 0.6330


In [15]:
print("\nBest Parameters:", best_params)
print("Best CV Accuracy:", best_score)

best_model.fit(X_train, y_train)

print("\nFinal model trained.")


Best Parameters: {'C': 0.1, 'gamma': 'scale', 'kernel': 'linear'}
Best CV Accuracy: 0.9945584737171083

Final model trained.


In [16]:
y_pred = best_model.predict(X_test)

test_accuracy = accuracy_score(y_test, y_pred)

print("\nTest Accuracy:", test_accuracy)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Test Accuracy: 0.9964317573595004

Confusion Matrix:
[[2727   13]
 [   7 2858]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2740
           1       1.00      1.00      1.00      2865

    accuracy                           1.00      5605
   macro avg       1.00      1.00      1.00      5605
weighted avg       1.00      1.00      1.00      5605



In [19]:
joblib.dump(best_model, "svm_model.pkl")

print(f"SVM model saved successfully at:\n{model_path}")

SVM model saved successfully at:
model\svm_model.pkl
